# 01 — Unconditional Signal Strength

Spearman IC for LM, LLM, OFI across 1/5/15-min horizons, both date windows.
Bootstrap CIs (n_boot=1000). BH FDR appended to secondary grid.

In [1]:
import sys, os
from pathlib import Path
for _cand in ['.', '..']:
    if (Path(_cand)/'src').is_dir() and (Path(_cand)/'legacy').is_dir():
        os.chdir(_cand); break
sys.path.insert(0, 'src'); sys.path.insert(0, 'legacy/src')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.stats_rigor import bootstrap_ic_ci, fdr_table, spearman_ic
from src.grid import append_cells
from src.report_io import save_fig, save_table, setup_style
from src.config import PANELS_2016, PANELS_2025

setup_style()
N_BOOT = 1000
HORIZONS = [1, 5, 15]
SCORERS  = ['lm_score', 'llm_score', 'ofi_z']
SCORER_LABELS = {'lm_score': 'LM', 'llm_score': 'LLM', 'ofi_z': 'OFI'}
print('Setup complete.')

Setup complete.


## 1. Load panels (both windows)

In [2]:
def load_pool(paths_dict):
    frames = []
    for ticker, path in paths_dict.items():
        if not Path(path).exists():
            print(f'  skip {ticker}: {path} not found')
            continue
        df = pd.read_csv(path)
        df['stock'] = ticker
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    pool = pd.concat(frames, ignore_index=True)
    print(f'  Loaded {len(pool):,} events from {len(frames)} tickers: {list(paths_dict)}')
    return pool

panel_2016 = load_pool(PANELS_2016)
panel_2025 = load_pool(PANELS_2025)
print(f'Window 2016-2020: {len(panel_2016):,} rows')
print(f'Window 2025:      {len(panel_2025):,} rows')

  Loaded 12,456 events from 6 tickers: ['AAPL', 'AMD', 'JPM', 'MU', 'NFLX', 'QCOM']
  Loaded 2,494 events from 5 tickers: ['AAPL', 'AMD', 'META', 'NVDA', 'TSLA']
Window 2016-2020: 12,456 rows
Window 2025:      2,494 rows


## 2. IC table with bootstrap CIs

In [3]:
def ic_table_with_ci(panel, window_label, n_boot=N_BOOT):
    rows = []
    for scorer in SCORERS:
        if scorer not in panel.columns:
            continue
        for h in HORIZONS:
            ret_col = f'ret_{h}m'
            if ret_col not in panel.columns:
                continue
            sig = panel[scorer]
            ret = panel[ret_col]
            ci  = bootstrap_ic_ci(sig, ret, n_boot=n_boot, seed=42)
            rows.append({
                'window': window_label,
                'scorer': SCORER_LABELS.get(scorer, scorer),
                'horizon': h,
                **ci,
            })
    return pd.DataFrame(rows)

tbl_2016 = ic_table_with_ci(panel_2016, '2016-2020')
tbl_2025 = ic_table_with_ci(panel_2025, '2025')
tbl_all  = pd.concat([tbl_2016, tbl_2025], ignore_index=True)

display_cols = ['window','scorer','horizon','ic','ci_lo','ci_hi','p_boot','n']
print(tbl_all[display_cols].to_string(index=False, float_format='{:.4f}'.format))

   window scorer  horizon      ic   ci_lo  ci_hi  p_boot     n
2016-2020     LM        1 -0.0088 -0.0267 0.0102  0.8080 12430
2016-2020     LM        5 -0.0094 -0.0262 0.0093  0.8430 12342
2016-2020     LM       15  0.0022 -0.0147 0.0207  0.4100 12115
2016-2020    LLM        1  0.0068 -0.0117 0.0247  0.2300 12430
2016-2020    LLM        5  0.0039 -0.0127 0.0216  0.3220 12342
2016-2020    LLM       15  0.0113 -0.0055 0.0304  0.1160 12115
2016-2020    OFI        1  0.0105 -0.0063 0.0280  0.1230 12430
2016-2020    OFI        5 -0.0097 -0.0278 0.0071  0.8680 12342
2016-2020    OFI       15 -0.0137 -0.0307 0.0032  0.9340 12115
     2025     LM        1 -0.0099 -0.0492 0.0307  0.6610  2480
     2025     LM        5 -0.0214 -0.0616 0.0161  0.8660  2462
     2025     LM       15  0.0319 -0.0073 0.0699  0.0610  2423
     2025    LLM        1  0.0031 -0.0365 0.0463  0.4450  2480
     2025    LLM        5  0.0337 -0.0066 0.0739  0.0520  2462
     2025    LLM       15  0.0059 -0.0325 0.0500  0.359

## 3. Null-result summary and bar figure

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, (window_label, tbl) in zip(axes, [('2016-2020', tbl_2016), ('2025', tbl_2025)]):
    for i, scorer in enumerate(SCORER_LABELS.values()):
        sub = tbl[tbl['scorer'] == scorer].sort_values('horizon')
        if sub.empty:
            continue
        x   = np.arange(len(sub))
        ics = sub['ic'].values
        lo  = ics - sub['ci_lo'].values
        hi  = sub['ci_hi'].values - ics
        offset = (i - 1) * 0.25
        ax.bar(x + offset, ics, width=0.22, label=scorer,
               yerr=[lo, hi], error_kw={'linewidth':0.8, 'capsize':3})
    ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.5)
    ax.set_xticks(range(len(HORIZONS)))
    ax.set_xticklabels([f'{h}-min' for h in HORIZONS])
    ax.set_title(f'Unconditional IC — {window_label}')
    ax.set_ylabel('Spearman IC')
    ax.legend(frameon=False)
    ax.grid(axis='y', alpha=0.25)

fig.suptitle('Unconditional signal IC (bars = 95% bootstrap CI)', y=1.02)
save_fig(fig, '01_unconditional_ic')
plt.close()

save_table(
    tbl_all[display_cols],
    '01_unconditional_ic',
    caption='Unconditional Spearman IC with 95\\% bootstrap CI (n\_boot=1000).',
    label='tab:unconditional_ic',
)

  Saved figure → results/figures/01_unconditional_ic.pdf
  Saved table  → results/tables/01_unconditional_ic.csv + results/tables/01_unconditional_ic.tex


PosixPath('results/tables/01_unconditional_ic.csv')

## 4. Append secondary cells to BH grid

In [5]:
grid_rows = []
for _, row in tbl_all.iterrows():
    grid_rows.append({
        'notebook': '01',
        'cell_id':  f"uncond_{row['window']}_{row['scorer']}_{row['horizon']}m",
        'stock':    'pooled',
        'scorer':   row['scorer'],
        'horizon':  row['horizon'],
        'n':        row['n'],
        'ic':       row['ic'],
        'ci_lo':    row['ci_lo'],
        'ci_hi':    row['ci_hi'],
        'p':        row['p_boot'],
    })

append_cells(grid_rows)
print(f'Appended {len(grid_rows)} cells to secondary grid.')

Appended 18 cells to secondary grid.


## 5. Takeaway

All unconditional IC values are small. This is expected — the hypothesis is
that the **interaction** signal on *confirmed* events (notebook 02) carries
the predictive content, not the raw scorers alone.